# Installing needed packages for 2m_miso

In [ ]:
from miso.hist_features import get_features
from miso.utils import *
from miso import Miso
from PIL import Image
import pandas as pd
import numpy as np
import scanpy as sc
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
Image.MAX_IMAGE_PIXELS = None
Image.MAX_IMAGE_PIXELS = None
import torch
import random

## Set up main run method (cpu/gpu)

**Parameters:** <br>
random.seed(seed=None) - Reseed the singleton RandomState instance. <br>
torch.manual_seed(seed) - Sets the seed for generating random numbers on all devices. Returns a torch.Generator object. <br>
random.seed(a=None, version=2) - Initialise the random number generator. <br>

torch.cuda.manual_seed(seed) - Set the seed for generating random numbers for the current GPU. <br>
torch.cuda.get_device_name(device=None) - Get the name of a device. <br>

In [ ]:
seed=100
np.random.seed(seed)
torch.manual_seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("CUDA is available. GPU:", torch.cuda.get_device_name(0))
else:
    device = 'cpu'
    print("CUDA is not available. Using CPU.")

## Reading joint Spatialmeta data (Y7_T_joint)

**Parameters:** <br>
scanpy.read_h5ad(filename) - Read .h5ad-formatted hdf5 file.

In [ ]:
os.chdir('miso_tutorial_data')

joint_adata_pp = sc.read_h5ad("../SpatialMETA/data/Y7_T_joint.h5ad")

print(joint_adata_pp)
print(joint_adata_pp.uns["spatial"].keys())

lib = list(joint_adata_pp.uns["spatial"].keys())[0]
print(joint_adata_pp.uns["spatial"][lib].keys())

print(joint_adata_pp.uns["spatial"][lib]["images"].keys())

## Load data and perform necessary preprocessing
Preprocessing provided by MISO. The only requirement is that the dataset be split into each modality. 

**Parameters:** <br>
preprocess(adata, modality) - Takes in a specific dataset and modality type, then preprocesses and returns the usable dataset. 

In [ ]:
rna = joint_adata_pp[:, joint_adata_pp.var["type"] == "ST"]
rna = preprocess(rna,modality='rna')
print(rna.shape)

protein = joint_adata_pp[:, joint_adata_pp.var["type"] == "SM"]
protein = preprocess(protein,modality='protein')
print(protein.shape)

## Running, training and saving the results of the MISO model 
If using a subset of modality-specific terms, the "ind_views" parameter should be a list with values that are entries to the indices of the modalities to be included, e.g.,  ind_views=[0,2] if including RNA and image features. <br>
If using a subset of interaction terms, the "combs" parameter should be a list of tuples with entries to the indices of the modalities for each interaction, e.g. combs = [(0,1),(0,2)] if including the RNA-protein and RNA-image interaction terms. <br>

**Parameters:** <br>
Miso(features, ind_views='all', combs='all', sparse=False, neighbors = None, device='cpu') - This is the step for preparing the multiple feature matrices for a learning model (model.train())<br><br>
model.train(self) - This step creates one neural network per modality (RNA, protein). Then learns the interactions between the features in both neural networks and condenses the final results into a single embedding matrix. <br><br>
np.save(file, arr, allow_pickle=True) - Save an array to a binary file in NumPy .npy format.

In [ ]:
model = Miso(
    [rna,protein],
    ind_views='all',
    combs='all',
    sparse=False,
    device=device)

model.train()

np.save('emb.npy', model.emb)

## Set up for the clustering on the histology image
This step first reads and stores the hires image from the joint dataset. Then the scaling factors of the correct image are read and stored. After that the coordinates of the spots are scaled to the image size for correct placement. The final step is storing all the spot data into a locs (location) pandas dataframe. <br>

**Parameters:** <br>
pd.DataFrame(data=None, index=None, columns=None, dtype=None, copy=None) - Two-dimensional, size-mutable, potentially heterogeneous tabular data.

In [ ]:
im = joint_adata_pp.uns["spatial"][lib]["images"]["hires"]

scale = (
    joint_adata_pp.uns["spatial"][lib]
    ["scalefactors"]["tissue_hires_scalef"]
)

coords = joint_adata_pp.obsm["spatial"] * scale

locs = pd.DataFrame(
    {
        "1": 1,                    # all spots in tissue
        "2": np.arange(len(coords)),
        "3": np.arange(len(coords)),
        "4": coords[:,1],          # row coordinate
        "5": coords[:,0],          # column coordinate
    },
    index=joint_adata_pp.obs_names
)

## Final histology image with clustering results
First the clusters are calculated by k-means. Then the clusters, spot coordinates, histology image, scale and size of dots (spots) are given to plot_on_histology. This method is from MISO and uses matplotlib to create the final image. <br>

**Parameters:**<br>
model.cluster(n_clusters) - Takes in a set amount of clusters. The method used is K-means clustering. This method returns the designated clusters in an array. <br><br>
plot_on_histology(clusters, locs, im, scale, s=10) - Takes in an array of clusters, the coordinates of spots, the histology image, the set scale and the size of the dots (spots). This method returns a histology image with clustered spots laid on top. 


In [ ]:
clusters = model.cluster(n_clusters=12)
scale = 1.0
plot_on_histology(clusters,locs,im,scale,s=10)